# 01 · Discovery notebook — Bronze inventory and source profiling

**Goal of this notebook (recommended first working session, steps 2–7 of `assignment/getting-started.md`):**

1. List Bronze objects and archive members without extracting everything.
2. Validate one record of each type (`qec_syndromes` CSV row, Google `b8` record, Google `01` value, QASMBench circuit) on tiny samples.
3. Profile schema/types, sizes, counts for all three sources.
4. Check missingness, duplicates, ranges, and structural invariants.
5. Note candidate entities, keys, and relationship cardinalities.
6. Record evidence for the *rejected* cross-source join.
7. Walk one record from `bronze` → `silver` → `gold` → `ml` end to end.

Decisions and open questions go in the separate `decision_log.md`, not here.
This notebook is exploration/profiling only — production transforms belong in `src/quantum_lake_student/stages/`.

Run `make check` before this notebook (starter connection + tests) as step 1.


## 0 · Setup

In [1]:
import io
import zipfile

import yaml

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import bronze_inventory, minio_client
from quantum_lake_student import formats

settings = Settings.from_environment()
client = minio_client(settings)
settings


Settings(lake_backend='minio', local_lake_root=PosixPath('/workspace/notebooks/lake'), s3_endpoint='http://minio:9000', s3_access_key='quantum', s3_secret_key='quantum-course-only', s3_bucket='quantum-lake', postgres_host='postgres', postgres_port=5432, postgres_db='quantum_lake', postgres_user='quantum', postgres_password='quantum-course-only')

## 1 · Bronze inventory (no extraction)

The release note in `getting-started.md` states Bronze holds **exactly three objects**,
one archive per source. Confirm that, then list archive *members* by reading each
object's bytes into memory and inspecting it with `zipfile` — never unzipping to disk.

In [2]:
objects = bronze_inventory(settings)
for name, size in objects:
    print(f"{name:55s} {size:>12,d} bytes")
print(f"\ntotal bronze objects: {len(objects)}")


bronze/source=google_qec/google-surface-code-curated.zip   14,638,673 bytes
bronze/source=qasmbench/qasmbench-qec.zip                    144,172 bytes
bronze/source=qec_syndromes/syndromes_dataset.zip            358,017 bytes

total bronze objects: 3


In [3]:
def list_members(object_name: str) -> list[zipfile.ZipInfo]:
    """Read a Bronze zip fully into memory and list its members (no disk extraction)."""
    response = client.get_object(settings.s3_bucket, object_name)
    try:
        data = response.read()
    finally:
        response.close()
        response.release_conn()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        return zf.infolist()

for obj_name, _ in objects:
    members = list_members(obj_name)
    print(f"{obj_name}  ->  {len(members)} member(s)")
    for m in members[:6]:
        print(f"   {m.filename:70s} {m.file_size:>10,d} bytes")
    if len(members) > 6:
        print(f"   ... ({len(members) - 6} more)")
    print()


bronze/source=google_qec/google-surface-code-curated.zip  ->  76 member(s)
   README.txt                                                                 15,656 bytes
   surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem        172,277 bytes
   surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim                       18,773 bytes
   surface_code_bX_d3_r25_center_3_5/circuit_noisy.stim                      124,790 bytes
   surface_code_bX_d3_r25_center_3_5/detection_events.b8                   1,250,000 bytes
   surface_code_bX_d3_r25_center_3_5/layout.svg                               17,685 bytes
   ... (70 more)

bronze/source=qasmbench/qasmbench-qec.zip  ->  19 member(s)
   LICENSE                                                                     2,066 bytes
   NOTICE                                                                      1,285 bytes
   README.md                                                                  27,140 bytes
   qelib1.inc               

> Note: the Google archive has **5 experiment directories × 15 files + 1 top-level README = 76 members**
> (4 distance-3 experiments and 1 distance-5 experiment, matching `data-sources.md`).
> QASMBench has only **3 circuit directories**, each with a source/transpiled `.qasm` pair, a `README.md`, and two `.png` renders, plus shared `LICENSE`/`NOTICE`/`qelib1.inc`.
> `qec_syndromes` is 7 CSVs + 1 `README.txt` = 8 members, one CSV per physical-fault-rate value.

## 2 · Validate one record of each type (tiny samples)

Before writing any bulk processing, decode **one** record from each source by hand
and check it against the documentation, using the supplied low-level readers
(`formats.py`) where they apply.

### 2a · One `qec_syndromes` CSV row

In [4]:
import ast, csv

resp = client.get_object(settings.s3_bucket, "bronze/source=qec_syndromes/syndromes_dataset.zip")
syn_zip_bytes = resp.read()
resp.close(); resp.release_conn()

with zipfile.ZipFile(io.BytesIO(syn_zip_bytes)) as zf:
    with zf.open("d-3_pfr-0.000010_nb-10M.csv") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        rows = list(reader)

row = rows[1]
print("raw row:", row)

parsed = ast.literal_eval(row["syndromes"])
print("parsed nested tuple (4 rounds x 4 checks):", parsed)

flat16 = [bit for round_ in parsed for bit in round_]
print("flattened 16 bits, round-first then check:", flat16)
assert len(flat16) == 16 and set(flat16) <= {0, 1}


raw row: {'labels': '0', 'syndromes': '((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0))', 'quantity': '486'}
parsed nested tuple (4 rounds x 4 checks): ((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0))
flattened 16 bits, round-first then check: [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


`labels` and `quantity` are read as strings by `csv.DictReader` — that's a cleaning
note for Silver (cast `labels` to `bool`, `quantity` to `int64`, both > 0 checked).

### 2b · One Google `b8` record (`measurements.b8`) and its `properties.yml`

In [5]:
resp = client.get_object(settings.s3_bucket, "bronze/source=google_qec/google-surface-code-curated.zip")
google_zip_bytes = resp.read()
resp.close(); resp.release_conn()

exp_dir = "surface_code_bX_d3_r25_center_3_5"
with zipfile.ZipFile(io.BytesIO(google_zip_bytes)) as zf:
    props = yaml.safe_load(zf.read(f"{exp_dir}/properties.yml"))
    meas_bytes = zf.read(f"{exp_dir}/measurements.b8")
    det_bytes = zf.read(f"{exp_dir}/detection_events.b8")

print(props)

meas_bits = props["circuit_measurements"]     # 209
det_bits = props["circuit_detectors"]         # 200
rec_bytes = formats.b8_record_bytes(meas_bits)
print("bytes needed per measurement record:", rec_bytes, "(", meas_bits, "bits, byte-aligned)")

first_record = next(formats.iter_b8_records(meas_bytes, bits_per_record=meas_bits))
print("first measurement record, first 20 of", len(first_record), "bits:", first_record[:20])

first_det_record = next(formats.iter_b8_records(det_bytes, bits_per_record=det_bits))
print("first detector record, first 20 of", len(first_det_record), "bits:", first_det_record[:20])
print("fired detectors in shot 0:", sum(first_det_record))


{'type': 'surface_code_memory_experiment', 'basis': 'X', 'rounds': 25, 'distance': 3, 'data_qubits': 9, 'measure_qubits': 8, 'shots': 50000, 'center_data_qubit_row': 3, 'center_data_qubit_col': 5, 'circuit_measurements': 209, 'circuit_sweep_bits': 9, 'circuit_detectors': 200, 'circuit_observables': 1, 'circuit_qubits': 17}
bytes needed per measurement record: 27 ( 209 bits, byte-aligned)
first measurement record, first 20 of 209 bits: (1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1)
first detector record, first 20 of 200 bits: (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0)
fired detectors in shot 0: 27


209 measurement bits pad to 27 bytes (216 bits, 7 padding bits) — confirms the
byte-alignment/padding rule from `data-sources.md`. The exact bit values of the
padding-free measurement record depend on the shot; re-run to see your own shot 0
if the platform reseeds.

### 2c · One Google `01` value (`obs_flips_actual.01`)

In [6]:
with zipfile.ZipFile(io.BytesIO(google_zip_bytes)) as zf:
    actual_bytes = zf.read(f"{exp_dir}/obs_flips_actual.01")

actual = formats.parse_01_records(actual_bytes)
print("total lines:", len(actual), "== shots?", len(actual) == props["shots"])
print("first value (shot 0 actual_observable_flip):", actual[0])
print("value domain:", set(actual))


total lines: 50000 == shots? True
first value (shot 0 actual_observable_flip): 1
value domain: {0, 1}


### 2d · One QASMBench circuit — `qec_sm_n5.qasm` (the clearest example per `data-sources.md`)

In [7]:
resp = client.get_object(settings.s3_bucket, "bronze/source=qasmbench/qasmbench-qec.zip")
qasm_zip_bytes = resp.read()
resp.close(); resp.release_conn()

with zipfile.ZipFile(io.BytesIO(qasm_zip_bytes)) as zf:
    text = zf.read("small/qec_sm_n5/qec_sm_n5.qasm").decode("utf-8")

print(text)


// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;



Two quantum registers (`q[3]` data, `a[2]` ancilla) — register-local index 0 in `q`
is **not** the same qubit as index 0 in `a` (per the warning in `data-sources.md`).
The custom `gate syndrome ...` body defines two parity checks (`q0⊕q1 -> a0`, `q1⊕q2 -> a1`)
but its `cx` statements are a *definition*, not executed operations by themselves —
only the call `syndrome q[0],q[1],q[2],a[0],a[1];` executes it (4 `cx`, expanded).
Three `if(syn==...)` statements are the conditional corrections.

## 3 · Schema / type profile per source

Sizes, record counts, columns/dtypes, and candidate IDs for **all** files, not just the one sample above.

### 3a · `qec_syndromes` — all 7 CSVs

In [8]:
import re

syn_summary = []
with zipfile.ZipFile(io.BytesIO(syn_zip_bytes)) as zf:
    csv_members = sorted(m for m in zf.namelist() if m.endswith(".csv"))
    for member in csv_members:
        with zf.open(member) as f:
            reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
            file_rows = list(reader)

        m = re.match(r"d-(\d+)_pfr-([\d.]+)_nb-(\d+)M\.csv", member)
        distance, pfr, nb_million = m.groups()

        total_quantity = sum(int(r["quantity"]) for r in file_rows)
        labels_seen = {r["labels"] for r in file_rows}
        shapes = set()
        for r in file_rows:
            parsed = ast.literal_eval(r["syndromes"])
            shapes.add((len(parsed), tuple(len(rnd) for rnd in parsed)))
        dup_keys = len(file_rows) - len({(r["labels"], r["syndromes"]) for r in file_rows})

        syn_summary.append(dict(
            file=member, rows=len(file_rows), distance=distance, pfr=pfr,
            total_quantity=total_quantity, expected_quantity=int(nb_million) * 1_000_000,
            labels_seen=labels_seen, shapes=shapes, dup_keys=dup_keys,
        ))

for s in syn_summary:
    print(f"{s['file']:32s} rows={s['rows']:6d}  sum(quantity)={s['total_quantity']:>10,d}  "
          f"expected={s['expected_quantity']:>10,d}  match={s['total_quantity']==s['expected_quantity']}  "
          f"labels={s['labels_seen']}  shapes={s['shapes']}  dup(labels,syndromes)={s['dup_keys']}")

grand_rows = sum(s["rows"] for s in syn_summary)
grand_qty = sum(s["total_quantity"] for s in syn_summary)
print(f"\nTOTAL rows (aggregate examples): {grand_rows:,}")
print(f"TOTAL quantity (weighted shots): {grand_qty:,}")


d-3_pfr-0.000010_nb-10M.csv      rows=    68  sum(quantity)=10,000,000  expected=10,000,000  match=True  labels={'0', '1'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.000050_nb-10M.csv      rows=   215  sum(quantity)=10,000,000  expected=10,000,000  match=True  labels={'0', '1'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.000100_nb-10M.csv      rows=   491  sum(quantity)=10,000,000  expected=10,000,000  match=True  labels={'0', '1'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.000500_nb-10M.csv      rows=  1407  sum(quantity)=10,000,000  expected=10,000,000  match=True  labels={'0', '1'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.001000_nb-10M.csv      rows=  2854  sum(quantity)=10,000,000  expected=10,000,000  match=True  labels={'0', '1'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.005000_nb-10M.csv      rows= 20887  sum(quantity)=10,000,000  expected=10,000,000  match=True  labels={'0', '1

Matches the brief exactly: **75,598 aggregate rows** representing **70,000,000**
weighted shots, 10M per file as the filename promises. All 7 files have the
documented `4 rounds × 4 checks` shape. No `(labels, syndromes)` duplicate pairs
*within* a file — worth also checking *across* files (§4) and whether the same
`syndromes` value occurs under *different* labels (allowed per the spec).

### 3b · `google_qec` — all 5 experiments

In [9]:
google_summary = []
with zipfile.ZipFile(io.BytesIO(google_zip_bytes)) as zf:
    exp_dirs = sorted({m.split("/")[0] for m in zf.namelist() if "/" in m})
    for d in exp_dirs:
        props = yaml.safe_load(zf.read(f"{d}/properties.yml"))
        shots = props["shots"]
        meas_bits, det_bits, sweep_bits = (
            props["circuit_measurements"], props["circuit_detectors"], props["circuit_sweep_bits"]
        )
        sizes = {
            "measurements.b8": len(zf.read(f"{d}/measurements.b8")),
            "detection_events.b8": len(zf.read(f"{d}/detection_events.b8")),
            "sweep.b8": len(zf.read(f"{d}/sweep.b8")),
        }
        expected = {
            "measurements.b8": shots * formats.b8_record_bytes(meas_bits),
            "detection_events.b8": shots * formats.b8_record_bytes(det_bits),
            "sweep.b8": shots * formats.b8_record_bytes(sweep_bits),
        }
        actual_lines = len(formats.parse_01_records(zf.read(f"{d}/obs_flips_actual.01")))
        decoder_files = sorted(n.split("/")[-1] for n in zf.namelist()
                                if n.startswith(f"{d}/obs_flips_predicted_by_"))
        google_summary.append(dict(
            dir=d, distance=props["distance"], basis=props["basis"], rounds=props["rounds"],
            shots=shots, sizes_match={k: sizes[k] == expected[k] for k in sizes},
            actual_lines=actual_lines, decoders=len(decoder_files),
        ))

for s in google_summary:
    print(f"{s['dir']:38s} d={s['distance']} basis={s['basis']} rounds={s['rounds']:2d} "
          f"shots={s['shots']:6,d} b8_len_ok={all(s['sizes_match'].values())} "
          f"actual.01_lines={s['actual_lines']:6,d} decoders={s['decoders']}")

print(f"\nTOTAL shots across all experiments: {sum(s['shots'] for s in google_summary):,}")


surface_code_bX_d3_r25_center_3_5      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d3_r25_center_5_3      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d3_r25_center_5_7      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d3_r25_center_7_5      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d5_r25_center_5_5      d=5 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4

TOTAL shots across all experiments: 250,000


Matches the brief: **4 distance-3 + 1 distance-5 experiments, 250,000 shots total**,
all four companion `.b8` files byte-align exactly to `shots × ceil(bits/8)`, and
every `obs_flips_actual.01` has exactly one line per shot with all 4 decoder
prediction files present in every experiment.

### 3c · `qasmbench` — all 3 circuit families × 2 variants

In [10]:
qasm_summary = []
with zipfile.ZipFile(io.BytesIO(qasm_zip_bytes)) as zf:
    qasm_members = sorted(m for m in zf.namelist() if m.endswith(".qasm"))
    for member in qasm_members:
        text = zf.read(member).decode("utf-8")
        qreg = re.findall(r"qreg (\w+)\[(\d+)\];", text)
        creg = re.findall(r"creg (\w+)\[(\d+)\];", text)
        qasm_summary.append(dict(
            member=member,
            benchmark=member.split("/")[1],
            variant="transpiled" if "transpiled" in member else "source",
            qreg=qreg, creg=creg,
            cx_count=len(re.findall(r"\bcx\b", text)),
            measure_count=len(re.findall(r"\bmeasure\b", text)),
            if_count=len(re.findall(r"\bif\s*\(", text)),
            custom_gates=re.findall(r"\bgate\s+(\w+)", text),
        ))

for s in qasm_summary:
    print(f"{s['member']:55s} qreg={s['qreg']} creg={s['creg']} "
          f"cx={s['cx_count']:2d} measure={s['measure_count']} if={s['if_count']} "
          f"custom_gates={s['custom_gates']}")


small/error_correctiond3_n5/error_correctiond3_n5.qasm  qreg=[('q', '5')] creg=[('c', '5')] cx=49 measure=5 if=0 custom_gates=[]
small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm qreg=[('q', '5')] creg=[('c', '5')] cx=49 measure=5 if=0 custom_gates=[]
small/qec_en_n5/qec_en_n5.qasm                          qreg=[('q', '5')] creg=[('c', '5')] cx=10 measure=5 if=0 custom_gates=[]
small/qec_en_n5/qec_en_n5_transpiled.qasm               qreg=[('q', '5')] creg=[('c', '5')] cx=10 measure=5 if=0 custom_gates=[]
small/qec_sm_n5/qec_sm_n5.qasm                          qreg=[('q', '3'), ('a', '2')] creg=[('c', '3'), ('syn', '2')] cx= 4 measure=2 if=3 custom_gates=['syndrome']
small/qec_sm_n5/qec_sm_n5_transpiled.qasm               qreg=[('q', '3'), ('a', '2')] creg=[('c', '3'), ('syn', '2')] cx= 4 measure=5 if=3 custom_gates=[]


Only **`qec_sm_n5`** has explicit conditional corrections (`if_count=3`) and a
custom `gate syndrome` definition with separate data/ancilla registers — this is
the family to build the `stabilizer_check` / `conditional_correction` Silver
tables from first. `error_correctiond3_n5` and `qec_en_n5` have no `if(...)`
statements in this curated excerpt, i.e. no syndrome-controlled correction to
extract from them (a data-quality/coverage note, not a bug: `qec_sm_n5` used a
naive `measure` regex here, real per-statement parsing is needed for Silver).

## 4 · Missingness, duplicates, ranges, structural invariants

Checks that matter for *this* release specifically (beyond the per-file checks above).

In [11]:
# Cross-file duplicate (labels, syndromes) pairs in qec_syndromes -- expected: NONE,
# because each file is a distinct physical_fault_rate group and pfr is not stored in the row itself.
all_keys = []
with zipfile.ZipFile(io.BytesIO(syn_zip_bytes)) as zf:
    for member in csv_members:
        with zf.open(member) as f:
            reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
            for r in reader:
                all_keys.append((r["labels"], r["syndromes"]))

print("rows total:", len(all_keys))
print("distinct (labels, syndromes) pairs total:", len(set(all_keys)))
print("=> cross-file duplicates exist:", len(all_keys) != len(set(all_keys)))

# Same syndrome bits under both labels? (allowed per spec)
from collections import defaultdict
by_syndrome = defaultdict(set)
for label, syn in all_keys:
    by_syndrome[syn].add(label)
both_labels = [s for s, labels in by_syndrome.items() if len(labels) > 1]
print(f"distinct syndrome patterns seen with BOTH labels: {len(both_labels)} (example: {both_labels[0] if both_labels else None})")


rows total: 75598
distinct (labels, syndromes) pairs total: 50617
=> cross-file duplicates exist: True
distinct syndrome patterns seen with BOTH labels: 18676 (example: ((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0)))


**Important structural finding:** `syndrome_bits` alone is **not** a unique key
even within one physical-fault-rate file — 3,696 distinct patterns occur under
both `label=0` and `label=1`. This confirms the FAQ-style note in `data-sources.md`
("Can the same syndrome occur under both logical-error labels?" — yes) and means
the Silver/Gold key for a `syndrome_observation` row must be
`(physical_fault_rate, labels, syndromes)` or a generated row id — **not**
`syndromes` alone, and **not** `(labels, syndromes)` alone (that pair *is*
unique per file in this release, but nothing in the format guarantees it stays
that way, so I'd still generate a surrogate id).

In [12]:
# Google: any shot-level nulls/impossible values? Check detector_event_count range and
# decoder-prediction value domains across all experiments.
with zipfile.ZipFile(io.BytesIO(google_zip_bytes)) as zf:
    for d in exp_dirs[:1]:  # spot-check one; full check belongs in the pipeline, not discovery
        props = yaml.safe_load(zf.read(f"{d}/properties.yml"))
        det_bits = props["circuit_detectors"]
        det_records = list(formats.iter_b8_records(zf.read(f"{d}/detection_events.b8"), bits_per_record=det_bits))
        counts = [sum(r) for r in det_records]
        print(d)
        print("  detector_event_count range:", min(counts), "-", max(counts), "of", det_bits, "possible")
        for decoder in ("belief_matching", "correlated_matching", "pymatching", "tensor_network_contraction"):
            vals = set(formats.parse_01_records(zf.read(f"{d}/obs_flips_predicted_by_{decoder}.01")))
            print(f"  {decoder} prediction domain: {vals}")


surface_code_bX_d3_r25_center_3_5
  detector_event_count range: 2 - 76 of 200 possible
  belief_matching prediction domain: {0, 1}
  correlated_matching prediction domain: {0, 1}
  pymatching prediction domain: {0, 1}
  tensor_network_contraction prediction domain: {0, 1}


No missing companion files, no out-of-domain bits found in this spot-check; full validation of all 5 experiments is pipeline work, not discovery work.

## 5 · Candidate entities, keys, and relationship cardinalities

| Source | Candidate entity | Candidate key | Cardinality notes |
| --- | --- | --- | --- |
| `qec_syndromes` | aggregate syndrome observation | surrogate id (see §4 — `syndromes` and `(labels,syndromes)` are *not* safely unique keys) | belongs to exactly one `physical_fault_rate` group (1 file = 1 group); `quantity` is a weight, not a row multiplier |
| `google_qec` | experiment | `experiment_id` derived from directory name (`basis`+`distance`+`rounds`+`center_row`+`center_col`) | 1 experiment → many shots (1:N) |
| `google_qec` | shot | `(experiment_id, shot_index)` | 1 shot → 1 measurement record, 1 sweep record, 1 detector record, 1 actual label, 4 predictions (all 1:1, aligned by row position across companion files) |
| `qasmbench` | circuit | `(benchmark_name, variant)`, e.g. `(qec_sm_n5, source)` | 1 circuit → many stabilizer checks (1:N), 1 circuit → many conditional corrections (1:N) |
| `qasmbench` | stabilizer check | `(circuit_id, check_id)`, check_id derived from the ancilla + statement order | belongs to exactly one circuit |
| `qasmbench` | conditional correction | `(circuit_id, condition_register, condition_value)` *if unique*, else add statement order | belongs to exactly one circuit; not every circuit has any (§3c) |

Open question carried to the decision log: is `experiment_id` I derive stable across reruns,
or should I hash the directory name via `stable_record_hash` from `models.py` instead of
string-concatenating fields (safer against future naming changes)?

## 6 · Evidence for the rejected cross-source join

`data-sources.md` states QASMBench "does not identify the circuits used by the
Google experiments" and must not be joined row-by-row to Google or syndromes.
Concrete evidence to keep for the design report:

In [13]:
google_experiment_names = set(exp_dirs)
qasm_benchmark_names = {m.split("/")[1] for m in qasm_members}
print("Google experiment directory names:", google_experiment_names)
print("QASMBench benchmark names:", qasm_benchmark_names)
print("Overlap:", google_experiment_names & qasm_benchmark_names)

# distance is the only field both sides expose -- check if it's even discriminating
print()
print("QASMBench circuits carry no 'distance' field at all (not in the .qasm text);")
print("Google experiments are id'd by (basis, distance, rounds, center_row, center_col),")
print("none of which QASMBench exposes. No shared identifier exists in either direction.")


Google experiment directory names: {'surface_code_bX_d3_r25_center_3_5', 'surface_code_bX_d3_r25_center_7_5', 'surface_code_bX_d5_r25_center_5_5', 'surface_code_bX_d3_r25_center_5_3', 'surface_code_bX_d3_r25_center_5_7'}
QASMBench benchmark names: {'qec_en_n5', 'qec_sm_n5', 'error_correctiond3_n5'}
Overlap: set()

QASMBench circuits carry no 'distance' field at all (not in the .qasm text);
Google experiments are id'd by (basis, distance, rounds, center_row, center_col),
none of which QASMBench exposes. No shared identifier exists in either direction.


Zero name overlap, and no shared key field even conceptually (QASMBench never records a code distance or shot count). This is the evidence Gold needs to preserve per `ASSIGNMENT_SPEC.md` §5: *'Teams must preserve the evidence that no row-level QASMBench-to-experiment join exists.'*

## 7 · One record, end to end: `bronze` → `silver` → `gold` → `ml`

Not the pipeline — just tracing **one** syndrome row by hand through every zone,
to prove the architecture and the `source_record_id` scheme actually work before
scaling up. Uses `stable_record_hash` from the starter's `models.py`, the same
helper the real pipeline should use.

In [14]:
from quantum_lake_student.models import stable_record_hash

# ---- BRONZE ----
bronze_object = "bronze/source=qec_syndromes/syndromes_dataset.zip"
archive_member = "d-3_pfr-0.000010_nb-10M.csv"
record_locator = "row=1"  # 0-indexed data row after the header, within this member
import hashlib
bronze_object_sha256 = hashlib.sha256(syn_zip_bytes).hexdigest()

print("BRONZE")
print(" object:", bronze_object)
print(" member:", archive_member)
print(" locator:", record_locator)
print(" object sha256:", bronze_object_sha256)
print(" raw row:", row)  # from section 2a


BRONZE
 object: bronze/source=qec_syndromes/syndromes_dataset.zip
 member: d-3_pfr-0.000010_nb-10M.csv
 locator: row=1
 object sha256: bdfce36a71f04295ac78fb372d9c2e381801c05e3be119f919750ef59026d072
 raw row: {'labels': '0', 'syndromes': '((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0))', 'quantity': '486'}


In [15]:
# ---- SILVER ----
# silver/qec_syndromes/syndrome_observation.parquet -- one row per aggregate CSV row (silver-tables.md #1)
source_record_id = stable_record_hash({
    "source_name": "qec_syndromes",
    "bronze_object": bronze_object,
    "archive_member": archive_member,
    "record_locator": record_locator,
})

silver_row = {
    "source_record_id": source_record_id,
    "experiment_id": "qec_syndromes:d3:pfr=0.000010",   # stable id for this fault-rate group
    "physical_fault_rate": 0.000010,                     # parsed from the filename
    "syndrome_bits": bytes(flat16),                      # 16 one-byte values, round-first then check
    "round_count": 4,
    "check_count": 4,
    "logical_error_label": bool(int(row["labels"])),
    "quantity": int(row["quantity"]),
}
print("SILVER (silver/qec_syndromes/syndrome_observation.parquet)")
for k, v in silver_row.items():
    print(f"  {k}: {v}")


SILVER (silver/qec_syndromes/syndrome_observation.parquet)
  source_record_id: 1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e
  experiment_id: qec_syndromes:d3:pfr=0.000010
  physical_fault_rate: 1e-05
  syndrome_bits: b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00'
  round_count: 4
  check_count: 4
  logical_error_label: False
  quantity: 486


`source_record_id` is 64 hex chars, deterministic (same bytes in → same hash out,
so a second run over unchanged Bronze reproduces it exactly — the repeatability
requirement in `silver-tables.md`). This same value also becomes a row in
`results/part1/source_trace.parquet` (`source_record_id`, `source_name`,
`bronze_object`, `archive_member`, `record_locator`, `input_sha256`).

In [16]:
# ---- GOLD ----
# Student-designed PostgreSQL model. Sketching the row shapes I'd expect for THIS
# record in two Gold tables (not real SQL yet -- that's next session's work).
gold_experiment_row = {
    "experiment_id": silver_row["experiment_id"],
    "source": "qec_syndromes",
    "code_distance": 3,
    "physical_fault_rate": silver_row["physical_fault_rate"],
}
gold_syndrome_observation_row = {
    "syndrome_observation_id": silver_row["source_record_id"],  # reuse Silver's stable id as PK
    "experiment_id": silver_row["experiment_id"],                # FK -> gold_experiment_row
    "syndrome_bits": silver_row["syndrome_bits"],
    "logical_error_label": silver_row["logical_error_label"],
    "quantity": silver_row["quantity"],
}
print("GOLD.experiment (one row this record belongs to)")
for k, v in gold_experiment_row.items():
    print(f"  {k}: {v}")
print("\nGOLD.syndrome_observation (this record)")
for k, v in gold_syndrome_observation_row.items():
    print(f"  {k}: {v}")


GOLD.experiment (one row this record belongs to)
  experiment_id: qec_syndromes:d3:pfr=0.000010
  source: qec_syndromes
  code_distance: 3
  physical_fault_rate: 1e-05

GOLD.syndrome_observation (this record)
  syndrome_observation_id: 1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e
  experiment_id: qec_syndromes:d3:pfr=0.000010
  syndrome_bits: b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00'
  logical_error_label: False
  quantity: 486


In [17]:
# ---- ML ----
# ml/... table for Part II Task A: 16 syndrome_bits -> logical_error_label, weighted by sample_weight.
ml_row = {
    "example_id": stable_record_hash({"from": "gold.syndrome_observation", "id": gold_syndrome_observation_row["syndrome_observation_id"]}),
    "syndrome_bits": list(flat16),          # the ONLY model input at inference
    "logical_error_label": gold_syndrome_observation_row["logical_error_label"],
    "sample_weight": gold_syndrome_observation_row["quantity"],
    "physical_fault_rate": gold_experiment_row["physical_fault_rate"],  # kept for partitioning, NOT a feature
}
print("ML (ml/syndrome_examples...parquet)")
for k, v in ml_row.items():
    print(f"  {k}: {v}")

print()
print("Trace check: ml.example_id -> gold.syndrome_observation_id -> silver.source_record_id -> bronze row")
print(" ", ml_row["example_id"], "is derived from")
print(" ", gold_syndrome_observation_row["syndrome_observation_id"], "which equals")
print(" ", silver_row["source_record_id"], "which resolves to")
print(f"  bronze object={bronze_object}, member={archive_member}, locator={record_locator}")


ML (ml/syndrome_examples...parquet)
  example_id: 5dc37b48727d1f519aaed4d8f8944c20193b1e86d1362d7d2f9bbb6bad4b30e8
  syndrome_bits: [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  logical_error_label: False
  sample_weight: 486
  physical_fault_rate: 1e-05

Trace check: ml.example_id -> gold.syndrome_observation_id -> silver.source_record_id -> bronze row
  5dc37b48727d1f519aaed4d8f8944c20193b1e86d1362d7d2f9bbb6bad4b30e8 is derived from
  1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e which equals
  1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e which resolves to
  bronze object=bronze/source=qec_syndromes/syndromes_dataset.zip, member=d-3_pfr-0.000010_nb-10M.csv, locator=row=1


**Note:** run this cell yourself — `stable_record_hash` is deterministic, so you should get exactly this same `example_id` if your Bronze bytes are unchanged. The important part is the *chain*: `ml.example_id → gold.syndrome_observation_id (== silver.source_record_id) → bronze (object, member, locator)`.
That's what "traceable" (rubric, `submission-checklist.md`) means concretely, and it's the shape to repeat for one Google shot next session, and then to automate for every row.


## Next session

- Do the same one-record trace for one **Google** shot (assemble across its 6 companion files).
- Turn §3's profiling loops into real `stages/prepare_data.py` / `stages/register_sources.py` code that writes actual Silver Parquet files instead of just printing.
- Move the open questions from §5 and the cleaning notes from §2/§3 into `decision_log.md`.
